# Reto 4 – Caso A: Prueba de hipótesis
**¿El Campus de Ingenierías (1) solicita en promedio más materiales digitales que los demás campus (2 y 3)?**

Codificación: 1 = Ingenierías, 2 = Ciencias Sociales y Humanidades, 3 = Ciencias de la Salud.

In [1]:
import pandas as pd
from scipy import stats

# Carga de datos (encabezados en la fila 10 del Excel)
df = pd.read_excel('Base_de_datos_3.xlsx', header=9)
df.columns = [str(c).strip() for c in df.columns]
df = df[['Campus', 'digitales']].dropna()
df['Campus'] = df['Campus'].astype(int)

ing   = df.loc[df['Campus'] == 1, 'digitales']
otros = df.loc[df['Campus'].isin([2, 3]), 'digitales']
df.head()

,Campus,digitales
0,1,6
1,2,7
2,1,8
3,2,9
4,2,7


## i. Hipótesis
- **H₀:** μ₁ ≤ μ₂ — Ingenierías no solicita en promedio más materiales digitales que los demás campus.
- **H₁:** μ₁ > μ₂ — Ingenierías solicita en promedio más materiales digitales (prueba unilateral derecha).

## ii. Supuestos: estadística descriptiva, normalidad y varianzas

In [2]:
for nombre, g in [('Ingenierías', ing), ('Demás campus', otros)]:
    print(f'{nombre:13s} n={len(g):5d}  media={g.mean():.4f}  s={g.std(ddof=1):.4f}  asimetría={stats.skew(g):.3f}')

lev = stats.levene(ing, otros)
print(f'\nLevene (igualdad de varianzas): W={lev.statistic:.4f}, p={lev.pvalue:.4f}')
print('n > 30 en ambos grupos -> por el Teorema del Límite Central la media muestral es aprox. normal')

Ingenierías   n= 3320  media=5.4099  s=2.4521  asimetría=0.051
Demás campus  n= 6680  media=5.5949  s=3.0513  asimetría=-0.058

Levene (igualdad de varianzas): W=449.9987, p=0.0000
n > 30 en ambos grupos -> por el Teorema del Límite Central la media muestral es aprox. normal


## iii. Nivel de significancia

In [3]:
alfa = 0.05
print('alfa =', alfa)

alfa = 0.05


## iv–v. Estadístico de prueba: t de Welch
Las varianzas difieren (Levene p < 0.05) y los grupos son desbalanceados, por lo que se usa la t de Welch.

In [4]:
t, p = stats.ttest_ind(ing, otros, equal_var=False, alternative='greater')
v1, v2, n1, n2 = ing.var(), otros.var(), len(ing), len(otros)
gl = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
t_crit = stats.t.ppf(1 - alfa, gl)
print(f't calculado = {t:.4f}')
print(f'grados de libertad = {gl:.2f}')
print(f't crítico (alfa=0.05) = {t_crit:.4f}')

t calculado = -3.2673
grados de libertad = 8029.77
t crítico (alfa=0.05) = 1.6450


## vi. Valor p

In [5]:
print(f'valor p = {p:.6f}')

valor p = 0.999455


## vii. Decisión sobre H₀

In [6]:
if p < alfa:
    print(f'p = {p:.6f} < {alfa}: SE RECHAZA H0. Ingenierías solicita más materiales digitales.')
else:
    print(f'p = {p:.6f} >= {alfa}: NO SE RECHAZA H0. No hay evidencia de que Ingenierías solicite más.')

p = 0.999455 >= 0.05: NO SE RECHAZA H0. No hay evidencia de que Ingenierías solicite más.


## viii. Conclusión
Con α = 0.05 no hay evidencia estadística de que el estudiantado de Ingenierías solicite en promedio más materiales digitales que el de los demás campus; su media muestral (5.41) es incluso ligeramente menor que la del resto (5.59).